In [ ]:
# ============================================================================
# GPU DETECTION AND TROUBLESHOOTING
# ============================================================================
import torch
import sys

print("=" * 60)
print("PYTORCH GPU DIAGNOSTICS")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"\nCUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU Detected!")
    print(f"  Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
    DEVICE = torch.device('cuda:0')
    print(f"\n✓ Using device: {DEVICE}")
else:
    print(f"\n✗ NO GPU DETECTED!")
    print("\nTROUBLESHOOTING:")
    print("1. Check if NVIDIA GPU driver is installed:")
    print("   Run: nvidia-smi (in terminal/cmd)")
    print("\n2. Check if PyTorch has CUDA support:")
    print(f"   torch.version.cuda: {torch.version.cuda}")
    if torch.version.cuda is None:
        print("   ✗ PyTorch was installed WITHOUT CUDA support!")
        print("\n   SOLUTION: Reinstall PyTorch with CUDA:")
        print("   Visit: https://pytorch.org/get-started/locally/")
        print("   Or run:")
        print("   pip uninstall torch torchvision")
        print("   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121")
    else:
        print(f"   ✓ CUDA version: {torch.version.cuda}")
        print("\n3. Check CUDA installation:")
        print("   Try: nvcc --version")
        print("\n4. Environment variables:")
        import os
        cuda_path = os.environ.get('CUDA_PATH', 'NOT SET')
        print(f"   CUDA_PATH: {cuda_path}")
    
    DEVICE = torch.device('cpu')
    print(f"\n⚠️  Falling back to CPU: {DEVICE}")

print("=" * 60)


In [ ]:
# ----------------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ----------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from torchvision.models import vgg19
from io import BytesIO

plt.switch_backend('Agg')

# Configuration Constants
SCALE_FACTOR = 4
BATCH_SIZE = 8
LR_SIZE = 96
HR_SIZE = LR_SIZE * SCALE_FACTOR
EPOCHS = 30

# Device (will be set by GPU detection cell, but define fallback here)
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Configuration loaded: Scale={SCALE_FACTOR}, Batch={BATCH_SIZE}, Device={DEVICE}")


In [ ]:
# ----------------------------------------------------------------------
# 2. REAL-WORLD DEGRADATION PIPELINE (Second-order degradation)
# ----------------------------------------------------------------------

def random_gaussian_kernel(kernel_size=7, sigma_min=0.2, sigma_max=3.0):
    """Generate random Gaussian blur kernel"""
    sigma = np.random.uniform(sigma_min, sigma_max)
    kernel = np.zeros((kernel_size, kernel_size))
    center = kernel_size // 2
    for i in range(kernel_size):
        for j in range(kernel_size):
            x, y = i - center, j - center
            kernel[i, j] = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    kernel = kernel / np.sum(kernel)
    return torch.FloatTensor(kernel).unsqueeze(0).unsqueeze(0)

def apply_blur(img):
    """Apply random Gaussian blur"""
    kernel_size = np.random.choice([5, 7, 9])
    kernel = random_gaussian_kernel(kernel_size)
    kernel = kernel.repeat(3, 1, 1, 1).to(img.device)
    
    padding = kernel_size // 2
    img_blurred = F.conv2d(img.unsqueeze(0), kernel, padding=padding, groups=3)
    return img_blurred.squeeze(0)

def apply_random_resize(img):
    """Apply random resize"""
    scale = np.random.uniform(0.5, 1.5)
    h, w = img.shape[1], img.shape[2]
    nh, nw = int(h * scale), int(w * scale)
    nh = max(2, nh)
    nw = max(2, nw)
    
    method = np.random.choice(['bilinear', 'bicubic', 'nearest'])
    img_resized = F.interpolate(img.unsqueeze(0), size=(nh, nw), mode=method, align_corners=False if method != 'nearest' else None)
    img_resized = F.interpolate(img_resized, size=(h, w), mode=method, align_corners=False if method != 'nearest' else None)
    return img_resized.squeeze(0)

def apply_noise(img):
    """Apply random noise"""
    std = np.random.uniform(0.0, 0.05)
    noise = torch.randn_like(img) * std
    return torch.clamp(img + noise, 0, 1)

def apply_jpeg_compression(img_tensor):
    """Simulate JPEG compression"""
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
    img_pil = Image.fromarray(img_np)
    
    quality = np.random.randint(60, 100)
    buffer = BytesIO()
    img_pil.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    img_compressed = Image.open(buffer)
    img_np = np.array(img_compressed).astype(np.float32) / 255.0
    return torch.FloatTensor(img_np).permute(2, 0, 1).to(img_tensor.device)

def degrade_once(img):
    """Apply one round of degradation: blur -> resize -> noise -> JPEG"""
    img = apply_blur(img)
    img = apply_random_resize(img)
    img = apply_noise(img)
    img = apply_jpeg_compression(img)
    return img

def generate_synthetic_lr(hr_img, scale=4):
    """Generate synthetic LR from HR using second-order degradation"""
    degraded = degrade_once(hr_img)
    degraded = degrade_once(degraded)  # Second order
    
    # Downscale to LR size
    h, w = hr_img.shape[1] // scale, hr_img.shape[2] // scale
    lr = F.interpolate(degraded.unsqueeze(0), size=(h, w), mode='area', align_corners=None)
    return lr.squeeze(0)


In [ ]:
# ----------------------------------------------------------------------
# 3. GENERATOR: Real-ESRGAN RRDB Architecture
# ----------------------------------------------------------------------

class ResidualDenseBlock(nn.Module):
    def __init__(self, num_filters=64, growth_channel=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_filters, growth_channel, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_filters + growth_channel, growth_channel, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_filters + 2 * growth_channel, growth_channel, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_filters + 3 * growth_channel, growth_channel, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_filters + 4 * growth_channel, num_filters, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        return x5

class ResidualInResidualDenseBlock(nn.Module):
    def __init__(self, num_filters=64, growth_channel=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_filters, growth_channel)
        self.rdb2 = ResidualDenseBlock(num_filters, growth_channel)
        self.rdb3 = ResidualDenseBlock(num_filters, growth_channel)
    
    def forward(self, x):
        out = self.rdb1(x)
        out = x + out * 0.2
        out = self.rdb2(out)
        out = x + out * 0.2
        out = self.rdb3(out)
        out = x + out * 0.2
        return out

def pixel_shuffle(x, scale_factor):
    b, c, h, w = x.size()
    out_channel = c // (scale_factor ** 2)
    h_out = h * scale_factor
    w_out = w * scale_factor
    x_view = x.view(b, out_channel, scale_factor, scale_factor, h, w)
    x_permuted = x_view.permute(0, 1, 4, 2, 5, 3).contiguous()
    return x_permuted.view(b, out_channel, h_out, w_out)

class RealESRGANGenerator(nn.Module):
    def __init__(self, scale=4, num_rrdb=16):
        super().__init__()
        self.conv_first = nn.Conv2d(3, 64, 3, 1, 1)
        self.body = nn.ModuleList([ResidualInResidualDenseBlock(64, 32) for _ in range(num_rrdb)])
        self.conv_body = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(64, 64 * 4, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(64, 64 * 4, 3, 1, 1) if scale == 4 else None
        self.conv_hr = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv_last = nn.Conv2d(64, 3, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        self.scale = scale
    
    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = feat
        for rrdb in self.body:
            body_feat = rrdb(body_feat)
        body_feat = self.conv_body(body_feat)
        feat = feat + body_feat
        
        if self.scale == 4:
            feat = self.lrelu(self.conv_up1(feat))
            feat = pixel_shuffle(feat, 2)
            feat = self.lrelu(self.conv_up2(feat))
            feat = pixel_shuffle(feat, 2)
        elif self.scale == 2:
            feat = self.lrelu(self.conv_up1(feat))
            feat = pixel_shuffle(feat, 2)
        
        feat = self.conv_hr(feat)
        out = self.conv_last(self.lrelu(feat))
        out = torch.tanh(out) * 0.58 + 0.5
        out = torch.clamp(out, 0, 1)
        return out


In [ ]:
# ----------------------------------------------------------------------
# 4. DISCRIMINATOR: U-Net Style
# ----------------------------------------------------------------------

class UNetDiscriminator(nn.Module):
    def __init__(self, num_in_ch=3):
        super().__init__()
        
        def conv_block(in_ch, out_ch, stride=1):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, stride, 1),
                nn.LeakyReLU(0.2, inplace=True)
            )
        
        # Down blocks
        self.down1 = nn.Sequential(conv_block(num_in_ch, 64), conv_block(64, 64))
        self.down2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.LeakyReLU(0.2, True), conv_block(128, 128))
        self.down3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.LeakyReLU(0.2, True), conv_block(256, 256))
        self.down4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1), nn.LeakyReLU(0.2, True), conv_block(512, 512))
        
        # Bottleneck
        self.bottleneck = nn.Sequential(conv_block(512, 512), conv_block(512, 512))
        
        # Up blocks
        self.up4 = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(512, 256, 3, 1, 1), nn.LeakyReLU(0.2, True))
        self.up4_conv = conv_block(512, 256)
        self.up3 = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(256, 128, 3, 1, 1), nn.LeakyReLU(0.2, True))
        self.up3_conv = conv_block(256, 128)
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(128, 64, 3, 1, 1), nn.LeakyReLU(0.2, True))
        self.up2_conv = conv_block(128, 64)
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(64, 64, 3, 1, 1), nn.LeakyReLU(0.2, True))
        self.up1_conv = conv_block(128, 64)
        
        self.final = nn.Conv2d(64, 1, 1)
    
    def forward(self, x):
        # Down
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        
        # Bottleneck
        b = self.bottleneck(d4)
        
        # Up with skip connections
        u4 = self.up4(b)
        u4 = self.up4_conv(torch.cat([u4, d3], 1))
        u3 = self.up3(u4)
        u3 = self.up3_conv(torch.cat([u3, d2], 1))
        u2 = self.up2(u3)
        u2 = self.up2_conv(torch.cat([u2, d1], 1))
        u1 = self.up1(u2)
        u1 = self.up1_conv(torch.cat([u1, x], 1))
        
        out = self.final(u1)
        return torch.mean(out.view(out.size(0), -1), dim=1, keepdim=True)


In [ ]:
# ----------------------------------------------------------------------
# 5. VGG PERCEPTUAL LOSS
# ----------------------------------------------------------------------

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(pretrained=True).features
        self.feature_extractor = nn.Sequential(*list(vgg.children())[:36])  # up to conv5_4
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    
    def forward(self, pred, target):
        # Normalize to VGG input range [0, 1] -> ImageNet normalization
        pred = (pred - self.mean) / self.std
        target = (target - self.mean) / self.std
        
        pred_features = self.feature_extractor(pred)
        target_features = self.feature_extractor(target)
        return F.l1_loss(pred_features, target_features)


In [ ]:
# This cell was intentionally left empty for spacing

In [ ]:
# ----------------------------------------------------------------------
# 6. DATASET CLASS
# ----------------------------------------------------------------------

class DIV2KDataset(Dataset):
    def __init__(self, hr_images, hr_size=384, lr_size=96, scale=4):
        self.hr_images = hr_images
        self.hr_size = hr_size
        self.lr_size = lr_size
        self.scale = scale
    
    def __len__(self):
        return len(self.hr_images)
    
    def __getitem__(self, idx):
        hr = self.hr_images[idx]
        
        # Random crop
        i = torch.randint(0, hr.shape[1] - self.hr_size + 1, (1,)).item()
        j = torch.randint(0, hr.shape[2] - self.hr_size + 1, (1,)).item()
        hr_patch = hr[:, i:i+self.hr_size, j:j+self.hr_size]
        
        # Random flip and rotate
        if torch.rand(1) > 0.5:
            hr_patch = torch.flip(hr_patch, [2])
        if torch.rand(1) > 0.5:
            hr_patch = torch.rot90(hr_patch, k=torch.randint(0, 4, (1,)).item(), dims=[1, 2])
        
        # Generate synthetic LR
        lr_patch = generate_synthetic_lr(hr_patch, scale=self.scale)
        
        return lr_patch, hr_patch

# Helper to create synthetic dataset if DIV2K is not available
def create_synthetic_dataset(num_samples=100):
    """Create synthetic HR images for testing"""
    hr_images = []
    for _ in range(num_samples):
        hr = torch.rand(3, HR_SIZE, HR_SIZE)
        hr_images.append(hr)
    return hr_images


In [ ]:
# ----------------------------------------------------------------------
# 7. RA-GAN LOSS FUNCTIONS
# ----------------------------------------------------------------------

def ragan_generator_loss(d_real, d_fake):
    """Relativistic average GAN generator loss"""
    mean_real = torch.mean(d_real)
    mean_fake = torch.mean(d_fake)
    return F.binary_cross_entropy_with_logits(d_fake - mean_real, torch.ones_like(d_fake))

def ragan_discriminator_loss(d_real, d_fake):
    """Relativistic average GAN discriminator loss"""
    mean_real = torch.mean(d_real)
    mean_fake = torch.mean(d_fake)
    loss_real = F.binary_cross_entropy_with_logits(d_real - mean_fake, torch.ones_like(d_real))
    loss_fake = F.binary_cross_entropy_with_logits(d_fake - mean_real, torch.zeros_like(d_fake))
    return loss_real + loss_fake


In [ ]:
# ----------------------------------------------------------------------
# 8. TRAINING LOOP
# ----------------------------------------------------------------------

def train_real_esrgan(generator, discriminator, dataloader, epochs=30, device=DEVICE):
    # Loss functions
    vgg_loss_fn = VGGPerceptualLoss().to(device)
    pixel_loss_fn = nn.L1Loss()
    
    # Optimizers
    g_optimizer = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.9, 0.99))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.9, 0.99))
    
    # Loss weights
    lambda_pixel = 1.0
    lambda_percep = 0.1
    lambda_adv = 0.005
    
    generator.train()
    discriminator.train()
    
    for epoch in range(epochs):
        g_loss_epoch = 0
        d_loss_epoch = 0
        
        for batch_idx, (lr, hr) in enumerate(dataloader):
            lr = lr.to(device)
            hr = hr.to(device)
            
            # Generate SR
            sr = generator(lr)
            
            # Update Discriminator
            d_optimizer.zero_grad()
            d_real = discriminator(hr)
            d_fake = discriminator(sr.detach())
            d_loss = ragan_discriminator_loss(d_real, d_fake)
            d_loss.backward()
            d_optimizer.step()
            
            # Update Generator
            g_optimizer.zero_grad()
            sr = generator(lr)
            
            # Generator losses
            pixel_loss = pixel_loss_fn(sr, hr)
            percep_loss = vgg_loss_fn(sr, hr)
            d_fake = discriminator(sr)
            d_real = discriminator(hr.detach())
            adv_loss = ragan_generator_loss(d_real, d_fake)
            
            g_loss = lambda_pixel * pixel_loss + lambda_percep * percep_loss + lambda_adv * adv_loss
            g_loss.backward()
            g_optimizer.step()
            
            g_loss_epoch += g_loss.item()
            d_loss_epoch += d_loss.item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx}/{len(dataloader)}], '
                      f'G_Loss: {g_loss.item():.4f}, D_Loss: {d_loss.item():.4f}')
        
        print(f'Epoch [{epoch+1}/{epochs}] - Avg G_Loss: {g_loss_epoch/len(dataloader):.4f}, '
              f'Avg D_Loss: {d_loss_epoch/len(dataloader):.4f}')
    
    return generator, discriminator


In [ ]:
# ----------------------------------------------------------------------
# 9. EVALUATION FUNCTIONS
# ----------------------------------------------------------------------

def calculate_psnr(img1, img2):
    """Calculate PSNR between two images"""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

def evaluate_model(generator, test_loader, device=DEVICE, num_samples=5):
    """Evaluate model on test set"""
    generator.eval()
    total_psnr = 0
    count = 0
    
    with torch.no_grad():
        for i, (lr, hr) in enumerate(test_loader):
            if i >= num_samples:
                break
            lr = lr.to(device)
            hr = hr.to(device)
            
            sr = generator(lr)
            psnr = calculate_psnr(sr, hr)
            total_psnr += psnr.item()
            count += 1
    
    generator.train()
    return total_psnr / count if count > 0 else 0


In [ ]:
# ----------------------------------------------------------------------
# 10. MAIN EXECUTION
# ----------------------------------------------------------------------

if __name__ == '__main__':
    print(f"\n{'='*60}")
    print("INITIALIZING REAL-ESRGAN TRAINING")
    print(f"{'='*60}\n")
    
    # Create dataset (using synthetic for now - replace with real DIV2K loading)
    print("Creating dataset...")
    hr_images = create_synthetic_dataset(num_samples=100)
    dataset = DIV2KDataset(hr_images, hr_size=HR_SIZE, lr_size=LR_SIZE, scale=SCALE_FACTOR)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    
    # Initialize models
    print("Initializing models...")
    generator = RealESRGANGenerator(scale=SCALE_FACTOR, num_rrdb=16).to(DEVICE)
    discriminator = UNetDiscriminator().to(DEVICE)
    
    print(f"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
    print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")
    print(f"\nStarting training on {DEVICE}...\n")
    
    # Train
    generator, discriminator = train_real_esrgan(
        generator, discriminator, dataloader, 
        epochs=EPOCHS, device=DEVICE
    )
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    
    # Save models
    torch.save(generator.state_dict(), 'real_esrgan_generator.pth')
    torch.save(discriminator.state_dict(), 'real_esrgan_discriminator.pth')
    print("Models saved!")
